## Bronze Ingestion
Reads raw GTFS `.txt` files from Volumes for each city and writes them as Delta tables in `gtfs_bronze`. \
Each table contains data from all cities, distinguished by the `city` column.

## Setup

In [0]:
%run ./00_config.py

## Imports

In [0]:
from pyspark.sql.functions import lit, current_timestamp
from functools import reduce
from datetime import datetime

## Bronze Ingestion

In [0]:
for table in GTFS_TABLES:
    dfs = []
    for city in CITIES.keys():
        # Skip calendar.txt for Roma — feed does not include it
        if city in CITIES_WITHOUT_CALENDAR and table == "calendar":
            continue
        file_path = f"{BASE_PATH}/{CITIES[city]}/{table}.txt"
        df = spark.read.csv(file_path, header=True, inferSchema=False)
        df = df.withColumn("city", lit(city)) \
            .withColumn("ingestion_ts", current_timestamp()) \
            .withColumn("source_file", lit(file_path))
        dfs.append(df)
    combined = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)
    combined.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(f"gtfs_bronze.{table}")
    count = spark.sql(f"SELECT COUNT(*) FROM gtfs_bronze.{table}").collect()[0][0]
    cities = spark.sql(f"SELECT city, COUNT(*) as cnt FROM gtfs_bronze.{table} GROUP BY city")

## Verification

In [0]:
for table in GTFS_TABLES:
    print(f"\n=== gtfs_bronze.{table} ===")
    display(spark.sql(f"""
        SELECT city, COUNT(*) as count 
        FROM gtfs_bronze.{table} 
        GROUP BY city 
        ORDER BY city
    """))